# Diabetes Health Indicators

First experiment using the shared workflow in `ml_workflow.py`.
The target is the binary indicator `Diabetes_binary`.

We use `diabetes_binary_health_indicators_BRFSS2015.csv`,
which retains the imbalanced class distribution.


In [1]:
import pandas as pd

from ml_workflow import (
    evaluate_classifier,
    load_diabetes,
    split_dataset,
    train_logistic_baseline,
)


## Dataset

We check the number of samples, the number of features, missing values,
and the class distribution.


In [2]:
X, y = load_diabetes()

print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Missing feature values: {X.isna().sum().sum()}")


Samples: 253680
Features: 21
Missing feature values: 0


In [3]:
class_distribution = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "proportion": y.value_counts(normalize=True).sort_index(),
})
class_distribution.index.name = "Diabetes_binary"
class_distribution.round(3)


,count,proportion
Diabetes_binary,,
0,218334,0.861
1,35346,0.139


## Train/test split

We use 80% of the data for training and 20% for testing.
The split is stratified to preserve similar class proportions.
The random seed is fixed at 67.


In [7]:
X_train, X_test, y_train, y_test = split_dataset(
    X, y, test_size=0.2, random_state=67
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Training samples: 202944
Test samples: 50736


## Logistic regression

The pipeline applies median imputation and standardization before
logistic regression. The preprocessing steps are fitted on the training split.


In [8]:
model = train_logistic_baseline(X_train, y_train)


## Test results

Precision, recall, and F1-score refer to the positive class (`1`).
We also calculate the test accuracy of always predicting the majority
class identified in the training split.


In [9]:
scores = evaluate_classifier(model, X_test, y_test)

majority_class = y_train.mode().iloc[0]
majority_accuracy = y_test.eq(majority_class).mean()
print(f"Majority-class accuracy: {majority_accuracy:.3f}")

results = pd.Series(scores, name="Logistic regression").to_frame()
results.round(3)


Majority-class accuracy: 0.861


,Logistic regression
accuracy,0.862
balanced_accuracy,0.565
precision,0.522
recall,0.153
f1,0.237
roc_auc,0.818
average_precision,0.394


The positive class represents 13.9% of the dataset. Always predicting the training majority class gives 86.1% accuracy on the test split. Logistic regression reaches 86.2% accuracy, with 56.5% balanced accuracy and 15.3% recall for the positive class.

The low recall shows that this first model misses most positive examples at its default decision threshold. Further experiments will compare class weighting and other classifiers using cross-validation on the training split.
